# CV Document Classification: Qualifications, Skills, Experience

This notebook loads the trained CV classifier model and applies it directly to CV documents (PDFs).

Given a resume/CV PDF, it will:
- Extract raw text from the document
- Clean and split the text into smaller segments
- Classify each segment as **Qualification**, **Skill**, or **Experience**
- Aggregate and display the results by category.


In [1]:
import pdfplumber
import re
import string
import torch
from collections import defaultdict

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("./cv_classifier_model")
tokenizer = AutoTokenizer.from_pretrained("./cv_classifier_model")


C:\Users\dehem\anaconda3\envs\projectdata\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import joblib
le = joblib.load("label_encoder.pkl")
id2label = dict(enumerate(le.classes_))

In [4]:
CONFIDENCE_THRESHOLD = 0.60  # adjust if needed

SECTION_KEYWORDS = {
    "Experience": [
        "work experience",
        "professional experience",
        "employment history",
        "experience"
    ],
    "Skill": [
        "skills",
        "technical skills",
        "technologies",
        "core competencies"
    ],
    "Qualification": [
        "education",
        "academic background",
        "qualification"
    ]
}

DEGREE_KEYWORDS = [
    "bsc", "msc", "phd",
    "bachelor", "master",
    "university", "college"
]


In [5]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text += page_text + "\n"
    return text


In [6]:
def clean_line(text: str) -> str:
    text = text.strip()
    if not text:
        return ""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.strip()

In [7]:
def split_into_lines(text: str):
    raw_lines = text.split("\n")
    merged_lines = []
    buffer = ""

    for line in raw_lines:
        line = line.strip()
        if not line:
            continue

        # If line looks like continuation (starts lowercase)
        if buffer and line and line[0].islower():
            buffer += " " + line
        else:
            if buffer:
                merged_lines.append(buffer.strip())
            buffer = line

    if buffer:
        merged_lines.append(buffer.strip())

    return merged_lines

In [8]:
def detect_section_header(line: str):
    line_lower = line.lower()
    for section, keywords in SECTION_KEYWORDS.items():
        for keyword in keywords:
            if keyword in line_lower:
                return section
    return None

In [9]:
def rule_based_classification(line: str):
    line_lower = line.lower()

    # If contains year → Experience
    if re.search(r"\b(19|20)\d{2}\b", line_lower):
        return "Experience"

    # Degree keywords → Qualification
    if any(word in line_lower for word in DEGREE_KEYWORDS):
        return "Qualification"

    # Comma-separated tech stack → Skill
    if "," in line and len(line.split(",")) >= 3 and " |" in line:
        return "Skill"

    return None

In [10]:
@torch.inference_mode()
def classify_line(text: str) -> str:
    if not text.strip():
        return "Other"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    confidence, pred_id = torch.max(probs, dim=-1)

    confidence = confidence.item()
    pred_id = pred_id.item()

    if confidence < CONFIDENCE_THRESHOLD:
        return "Other"

    return id2label.get(pred_id, "Other")


In [11]:
def looks_like_skill_line(line):
    tech_keywords = []
    nlp_test = spacy.load("skill_ner_model")
    doc = nlp_test("pandas")
    for ent in doc.ents:
        tech_keywords.append(ent.text)
    
    count = count(tech_keywords)
    
    return count >= 2

In [12]:
from collections import defaultdict

def classify_document(pdf_path: str):
    raw_text = extract_text_from_pdf(pdf_path)
    lines = split_into_lines(raw_text)

    results = defaultdict(list)
    current_section = None

    for line in lines:
        cleaned = clean_line(line)
        if not cleaned:
            continue

        # Detect section header
        detected_section = detect_section_header(line)
        if detected_section:
            current_section = detected_section
            continue

        #  Rule-based override first
        rule_label = rule_based_classification(line)
        if rule_label:
            results[rule_label].append(line.strip())
            continue

        # If inside a section → trust structure
        if current_section:
            results[current_section].append(line.strip())
            continue

        #  ML fallback (only if no structure + no rule)
        label = classify_line(cleaned)
        results[label].append(line.strip())

    return results

In [13]:
def print_results(results):
    for section, lines in results.items():
        print(f"\n=== {section} ({len(lines)} lines) ===")
        for line in lines:
            print("-", line)

In [64]:
results = classify_document("resume_pdfs/resume_00005_Python_Developer.pdf")


## Ranking for the CV based on the JD 

In [221]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model_embedding = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


In [222]:
# def get_embedding(text):
#     return model_embedding.encode(text, convert_to_numpy=True)

# def similarity(text1, text2):
#     emb1 = get_embedding(text1)
#     emb2 = get_embedding(text2)
#     return cosine_similarity([emb1], [emb2])[0][0]

In [223]:
cv  = {
   "name":"name 01",
    "Experience": " ".join(results['Experience']),
    "Skill": " ".join(results['Skill']),
    "Qualification" : " ".join(results['Qualification'])
}

In [268]:
job = {
       "Skill":"""
        Python,
        Django,
        Flask,
        REST APIs,
        SQL,
        PostgreSQL,
        Git,
        Docker,
        AWS,
        Unit Testing,
        OOP,
        Data Structures,
        Linux
    """,

    "Qualification": """
    Bachelor’s or Master’s degree in Computer Science,
    Software Engineering, Information Technology, or related field.
    """,

    "Experience": """
    Minimum 3+ years of experience in Python development.
    Experience building RESTful APIs using Django or Flask.
    Strong knowledge of databases such as PostgreSQL or MySQL.
    Experience with Git version control and CI/CD pipelines.
    Experience working in Linux environments.
    """
}

In [269]:
job_exp_emb = model_embedding.encode(job["Experience"], convert_to_numpy=True)
job_skill_emb = model_embedding.encode(job["Skill"], convert_to_numpy=True)
job_qual_emb = model_embedding.encode(job["Qualification"], convert_to_numpy=True)

In [270]:
def rank_candidates(cvs):
    scores = []

    for cv in cvs:
        score = compute_score(cv)
        scores.append((cv["name"], score))

    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    return ranked

In [271]:
def split_skills(text):
    separators = [",", "?", "-", ".", "\n"]
    for sep in separators:
        text = text.replace(sep, "|")
    return [s.strip() for s in text.split("|") if len(s.strip()) > 2]

In [272]:
cv_skills = split_skills(cv['Skill'])
job_required_skills = split_skills(job['Skill'])

In [273]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def semantic_skill_score(cv_embs, job_embs):
    cv_embs = model_embedding.encode(cv_embs, convert_to_numpy=True)
    job_embs = model_embedding.encode(job_embs, convert_to_numpy=True)
    total = 0

    for job_emb in job_embs:

        # Ensure 2D shape
        job_emb = np.array(job_emb).reshape(1, -1)
        cv_embs = np.array(cv_embs)

        sims = cosine_similarity(job_emb, cv_embs)[0]

        total += np.max(sims)

    return total / len(job_embs)

In [274]:
def qualification_semantic_score(cv_qual, job_qual):

    if not cv_qual or not job_qual:
        return 0

    cv_emb = model_embedding.encode(
        cv_qual,
        convert_to_numpy=True
    )

    job_emb = model_embedding.encode(
        job_qual,
        convert_to_numpy=True
    )

    score = cosine_similarity(
        cv_emb.reshape(1,-1),
        job_emb.reshape(1,-1)
    )[0][0]

    return max(0, min(score, 1))

In [275]:
def experience_semantic_score(cv_exp_text, job_exp_text):

    if not cv_exp_text or not job_exp_text:
        return 0

    cv_emb = model_embedding.encode(cv_exp_text, convert_to_numpy=True)
    job_emb = model_embedding.encode(job_exp_text, convert_to_numpy=True)

    semantic_score = cosine_similarity(
        cv_emb.reshape(1,-1),
        job_emb.reshape(1,-1)
    )[0][0]

    return max(0, min(semantic_score, 1))

In [276]:
def experience_year_bonus(cv_years, required_years):

    if cv_years >= required_years:
        return 1
    else:
        return (cv_years / required_years) 

In [277]:
import re
from datetime import datetime

def extract_experience_years(text):

    # Pattern for Month Year to Month Year / Present
    pattern = r'([A-Za-z]+\s\d{4})\s*to\s*(Present|[A-Za-z]+\s\d{4})'
    
    matches = re.findall(pattern, text)

    total_months = 0
    current_date = datetime.now()

    for start, end in matches:
        try:
            start_date = datetime.strptime(start, "%B %Y")
        except:
            try:
                start_date = datetime.strptime(start, "%b %Y")
            except:
                continue

        if end == "Present":
            end_date = current_date
        else:
            try:
                end_date = datetime.strptime(end, "%B %Y")
            except:
                try:
                    end_date = datetime.strptime(end, "%b %Y")
                except:
                    continue

        months = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month)
        total_months += max(0, months)

    total_years = total_months / 12

    return round(total_years, 2)

In [278]:
def degree_level_score(cv_qualification):

    text = cv_qualification.lower()

    if "phd" in text or "doctor" in text:
        return 1.0
    elif "master" in text or "msc" in text or "mtech" in text:
        return 0.9
    elif "bachelor" in text or "bsc" in text or "btech" in text:
        return 0.8
    elif "diploma" in text:
        return 0.4
    else:
        return 0.0

In [279]:
import re

def extract_required_years(job_text):

    text = job_text.lower()

    # Pattern 1: "5+ years"
    match = re.search(r'(\d+)\s*\+\s*years?', text)
    if match:
        return int(match.group(1))

    # Pattern 2: "minimum 3 years" or "at least 3 years"
    match = re.search(r'(minimum|at least)\s*(\d+)\s*years?', text)
    if match:
        return int(match.group(2))

    # Pattern 3: "3-5 years"
    match = re.search(r'(\d+)\s*-\s*(\d+)\s*years?', text)
    if match:
        return int(match.group(1))  # take lower bound

    # Pattern 4: "3 years experience"
    match = re.search(r'(\d+)\s*years?', text)
    if match:
        return int(match.group(1))

    # Default if nothing found
    return 0

In [280]:
print("Skill Semantic score:",semantic_skill_score(cv_skills,job_required_skills))

Skill Semantic score: 0.80379987


In [281]:
job_required_skills

['Python',
 'Django',
 'Flask',
 'REST APIs',
 'SQL',
 'PostgreSQL',
 'Git',
 'Docker',
 'AWS',
 'Unit Testing',
 'OOP',
 'Data Structures',
 'Linux']

In [282]:
def rank_candidate(cv, job):

    # -----------------------------
    #  Skill Score
    # -----------------------------
    skill_score = semantic_skill_score(
        split_skills(cv["Skill"]),
        split_skills(job["Skill"])
    )

    # -----------------------------
    #  Experience Score
    # -----------------------------
    cv_years = extract_experience_years(cv["Experience"])

    job_years = extract_required_years(job["Experience"])  # better regex function

    year_score = experience_year_bonus(
        cv_years,
        job_years
    )

    exp_semantic = experience_semantic_score(
        cv["Experience"],
        job["Experience"]
    )

    final_exp_score = 0.6 * exp_semantic + 0.4 * year_score

    # -----------------------------
    # Qualification Scores
    # -----------------------------
    degree_score = degree_level_score(
        cv["Qualification"]
    )

    qualification_score = qualification_semantic_score(
        cv["Qualification"],
        job["Qualification"]
    )

    final_qualification_score = (
        0.3 * qualification_score +
        0.7 * degree_score
    )
    # -----------------------------
    # Final Weighted Score
    # -----------------------------
    final_score = (
        0.30 * skill_score +
        0.40 * final_exp_score +
        0.30 * final_qualification_score
    )

    return {
        "final_score": round(final_score, 4),
        "skill_score": round(skill_score, 4),
        "experience_score": round(final_exp_score, 4),
        "qualification_score": round(final_qualification_score, 4),
        "years_experience": cv_years
    }

In [283]:
cv_list = []

for i in range(10, 30):

    doc = classify_document(
        f"resume_pdfs/resume_000{i}_Python_Developer.pdf"
    )

    my_doc = {
        "Skill": " ".join(doc["Skill"]),
        "Qualification": " ".join(doc["Qualification"]),
        "Experience": " ".join(doc["Experience"])
    }

    cv_list.append(my_doc)

In [284]:
rank_candidate(cv,job)

{'final_score': np.float32(0.7008),
 'skill_score': np.float32(0.8038),
 'experience_score': np.float32(0.6614),
 'qualification_score': np.float32(0.6503),
 'years_experience': 14.25}

In [287]:
ranked_results = sorted(
    [
        {
            "candidate_id": i+1,
            "score": rank_candidate(cv, job)["final_score"]
        }
        for i, cv in enumerate(cv_list)
    ],
    key=lambda x: x["score"],
    reverse=True
)

In [286]:
for i in ranked_results: 
    print("Candidate id :",i['candidate_id']," Score:",i['score'])

Candidate id : 4  Score: 0.7442
Candidate id : 9  Score: 0.705
Candidate id : 20  Score: 0.6727
Candidate id : 13  Score: 0.646
Candidate id : 10  Score: 0.6434
Candidate id : 3  Score: 0.634
Candidate id : 5  Score: 0.6037
Candidate id : 15  Score: 0.5959
Candidate id : 8  Score: 0.584
Candidate id : 7  Score: 0.5376
Candidate id : 16  Score: 0.5276
Candidate id : 12  Score: 0.5021
Candidate id : 18  Score: 0.4934
Candidate id : 6  Score: 0.4728
Candidate id : 17  Score: 0.4591
Candidate id : 11  Score: 0.3974
Candidate id : 2  Score: 0.3867
Candidate id : 14  Score: 0.3826
Candidate id : 19  Score: 0.3716
Candidate id : 1  Score: 0.3512
